# 21 — Direction 1, step 6: train the policy through the replica (SHAC), score it, package it for submission

One GPU T4 ×2 session (`hvac_controller/d1/train.py`, `policy.py`, `evaluate.py`):

1. **smoke** (≈ 5 min, GPU 0): layout of the scenario against its simulator objects, parity of the submission path
   (`KagglePolicy` on the organisers' simulator, CPU) with the training path (GPU), CUDA graph against eager, a short
   run. Stops the notebook if anything fails.
2. **training**, both GPUs at once, `MINUTES` each:
   - `held_out` (GPU 0): the 25 scenarios of the other buildings; checkpoints chosen on `HELD_OUT`'s 5 scenarios in the
     replica. The generalisation evidence.
   - `all` (GPU 1): all 30 scenarios; checkpoint chosen on them in the replica. **The submission.**
3. **original simulator** (4 CPUs): `held_out`'s policy on the 5 held-out scenarios and `all`'s on atlanta_2 (the
   public example), 30 days each, against K10 from notebook 20.
4. **package**: `/kaggle/working/data.zip`, flat: `d1_policy.pt` (the `all` run's best), `d1_policy.py` (the policy
   code, NumPy + torch only), `README.txt`. Download it, upload it as a Kaggle dataset; notebook 22 submits it.

Settings from the pilot (2026-10-20): h = 288, γ = 0.998 (0.00285 on atlanta_2 against K10's 0.00716); run commands
on in the comfort hours, learnable outside them only; collapse guard; eager (with 5–6 buildings per update the CUDA
graphs of every building do not fit together).

**Setup:** accelerator **GPU T4 ×2**; Internet on (clones the repo); attach two inputs: the competition data and the
dataset made from notebook 20's outputs (`data/data.zip`: 30 replica caches + K10 results). ≈ 7 h with `MINUTES = 300`.
**Output:** `/kaggle/working/data.zip` (the submission dataset), `/kaggle/working/d1_train_full.zip` (logs, reports, both
policies, original-simulator results).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
HELD_OUT = "building_train_2"
SETTINGS = {"h": 288, "gamma": 0.998, "graph": 0, "eval_minutes": 60, "eval_seeds": 1}
MINUTES = 300             # training budget of each run
SMOKE = "building_train_1__atlanta_2"

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch
N_GPU = torch.cuda.device_count()
assert N_GPU >= 2, "set the accelerator to GPU T4 x2"
print(torch.__version__, N_GPU, "x", torch.cuda.get_device_name())

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)
sys.path.insert(0, str(REPO))

ALL = sorted(f"{p.parent.parent.name}__{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))
CACHES = {p.stem: p for p in sorted(Path("/kaggle/input").rglob("*.pkl"))}
K10 = sorted(Path("/kaggle/input").rglob("original__*__k10__s*.json"))
missing = [s for s in ALL if s not in CACHES]
assert not missing, f"replica caches missing (attach the data.zip dataset): {missing}"
print(len(ALL), "scenarios,", len(CACHES), "caches,", len(K10), "K10 results")
RUNS = {"held_out": ([s for s in ALL if not s.startswith(HELD_OUT + "__")], [s for s in ALL if s.startswith(HELD_OUT + "__")]),
        "all": (ALL, [])}
OUT = Path("/kaggle/working/d1_train_full")
OUT.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(REPO), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")


def train_cmd(out_dir, train, val, settings, extra=()):
    cmd = [sys.executable, "-m", "hvac_controller.d1.train", str(out_dir), "--caches", ",".join(str(CACHES[s]) for s in train)]
    if val:
        cmd += ["--val-caches", ",".join(str(CACHES[s]) for s in val)]
    for k, v in settings.items():
        cmd += [f"--{k.replace('_', '-')}", str(v)]
    return cmd + list(extra)


def tail(path, n=3000):
    text = Path(path).read_text() if Path(path).exists() else ""
    return text[-8000:] if "Traceback" in text else text[-n:]

In [ ]:
# 1. Smoke: start-up checks and a two-minute run on GPU 0. Stops here if anything is wrong.
smoke = OUT / "smoke"
smoke.mkdir(exist_ok=True)
t0 = time.time()
with open(smoke / "log.txt", "w") as log:
    rc = subprocess.run(train_cmd(smoke, [SMOKE], [], {"h": 36, "minutes": 2, "eval_minutes": 100, "eval_steps": 288},
                                  ["--smoke"]), cwd=REPO, env=dict(ENV, CUDA_VISIBLE_DEVICES="0"), stdout=log,
                        stderr=subprocess.STDOUT).returncode
print(f"smoke exit {rc} in {time.time() - t0:.0f}s\n" + tail(smoke / "log.txt"))
rep = json.loads((smoke / "report.json").read_text()) if (smoke / "report.json").exists() else {}
assert rc == 0 and rep.get("parity", {}).get("ok"), "smoke failed: read the log above"

In [ ]:
# 2. The two training runs, one per GPU; evaluations and collapses are echoed as they happen.
t0, procs = time.time(), {}
for gpu, (name, (train, val)) in enumerate(RUNS.items()):
    d = OUT / name
    d.mkdir(exist_ok=True)
    log = open(d / "log.txt", "w")
    procs[name] = (subprocess.Popen(train_cmd(d, train, val, {**SETTINGS, "minutes": MINUTES}), cwd=REPO, stdout=log,
                                    stderr=subprocess.STDOUT, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))), log, d)
    print(f"GPU {gpu}: {name}: {len(train)} train scenarios, {len(val)} held out", flush=True)
seen = {name: 0 for name in procs}
while True:
    running = any(p.poll() is None for p, _, _ in procs.values())
    for name, (p, _, d) in procs.items():
        lines = (d / "log.txt").read_text().splitlines()
        for line in lines[seen[name]:]:
            if line.startswith(("[eval]", "Traceback")) or "collapse" in line or "building(s)" in line:
                print(f"[{(time.time() - t0) / 60:5.0f} min] {name}: {line[:400]}", flush=True)
        seen[name] = len(lines)
    if not running:
        break
    time.sleep(60)
REPORTS = {}
for name, (p, log, d) in procs.items():
    log.close()
    REPORTS[name] = json.loads((d / "report.json").read_text()) if (d / "report.json").exists() else {}
    print(f"===== {name}: exit {p.returncode}, best {REPORTS[name].get('best')}, {REPORTS[name].get('iterations')} iterations,"
          f" rollbacks {REPORTS[name].get('rollbacks')}\n" + tail(d / "log.txt", 2500))

In [ ]:
# 3. The organisers' simulator: held_out's policy on the held-out scenarios, all's on atlanta_2; against K10.
from hvac_controller.d1 import evaluate as ev

os.environ.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", CUDA_VISIBLE_DEVICES="",
                  PYTHONPATH=str(REPO))
os.chdir(REPO)
EVAL = OUT / "original"
EVAL.mkdir(exist_ok=True)
for p in K10:
    shutil.copy(p, EVAL / p.name)
jobs = []
for name, scenarios in (("held_out", RUNS["held_out"][1]), ("all", [SMOKE])):
    policy = OUT / name / "d1_policy.pt"
    if not policy.exists():
        print(f"{name}: no policy saved, skipped")
        continue
    jobs += [{"where": "original", "scenario": s.replace("__", "/", 1), "policy": f"trained:{policy}", "seed": 0,
              "steps": 8640, "config": str(ROOT / s.replace("__", "/", 1) / "config.gin"), "cache": None} for s in scenarios]
results = ev.run_jobs(jobs, EVAL, 4)
ens = ev.ensembles(results)
for name in ("held_out", "all"):
    cmp = ev.compare(ens, name, "k10")
    print(f"\n{name} in the original vs K10:")
    for scenario, c in cmp.items():
        if scenario == "mean":
            print(f"  mean: {c['candidate']:.5f} vs K10 {c['null']:.5f} ({c['candidate'] / c['null'] - 1:+.1%})")
            continue
        e = ens[("original", scenario, name)]
        print(f"  {scenario:30s} {c['candidate']:.5f} vs {c['null']:.5f} ({c['rel']:+.1%}, z {c['z']:+.1f}) | "
              f"regret {e['regret']:.4f} cost {e['cost']:.4f} carbon {e['carbon']:.4f}")

In [ ]:
# 4. Package the submission dataset (flat data.zip) and the logs.
readme = (f"Direction 1 policy, commit {COMMIT}, trained on all 30 shipped scenarios (best replica score "
          f"{REPORTS.get('all', {}).get('best')}). d1_policy.py = hvac_controller/d1/policy.py. Used by notebook 22.\n")
with zipfile.ZipFile("/kaggle/working/data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT / "all" / "d1_policy.pt", "d1_policy.pt")
    z.write(REPO / "hvac_controller" / "d1" / "policy.py", "d1_policy.py")
    z.writestr("README.txt", readme)
print("wrote /kaggle/working/data.zip:", zipfile.ZipFile("/kaggle/working/data.zip").namelist())
name = "/kaggle/working/d1_train_full.zip"
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and (p.suffix != ".pt" or p.name == "d1_policy.pt"):
            z.write(p, p.relative_to(OUT.parent))
print("wrote", name)